# Fabric Gateway Inventory → `lh_fabric_management`

Enumerates every **gateway** visible in the tenant and maps each one to the
**workspaces** and **semantic models** that use it, then writes the result to
Delta tables in the **`lh_fabric_management`** lakehouse.

## Tables written
| Table | Grain | Contents |
|---|---|---|
| `gateways` | one row per gateway cluster | Cluster metadata, primary member status/version, contact, counts |
| `gateway_members` | one row per member gateway | Machine, version, status, contact for each cluster member |
| `gateway_admins` | one row per admin principal | Which users / apps administer each gateway |
| `gateway_datasources` | one row per gateway-bound data source | The distinct data sources each gateway fronts |
| `gateway_semantic_model_map` | one row per (semantic model × gateway data source) | Which model, in which workspace, uses which gateway |

## How the mapping is built
```
gateway ─(gatewayId)→ datasource instance ─(datasourceInstanceId)→ semantic model ─(lives in)→ workspace
```
Gateway metadata comes from the **admin gateway-clusters endpoint**
(`GET /v2.0/myorg/gatewayclusters` — the tenant-wide list that backs *Manage
connections and gateways ▸ Tenant administration*; undocumented but stable).
The model ↔ gateway links come from the Power BI **metadata scanner**
(`getInfo` → `scanResult`): its top-level `datasourceInstances` carry the
`gatewayId`, and each dataset's `datasourceUsages` reference those instances by id.

## Prerequisites
1. **`lh_fabric_management` exists in this workspace and is schema-enabled.** Tables are written to the **`fabricmanagement`** schema (`Tables/fabricmanagement/…`). The lakehouse does *not* need to be attached as the default — the notebook resolves its OneLake path and writes there directly (attach it only if you want to browse the tables in the Explorer). For a classic (non-schema) lakehouse, set `LAKEHOUSE_SCHEMA = None`. If it lives in another workspace, pass `workspaceId=` to `notebookutils.lakehouse.get` in the write cell.
2. The run identity must be a **Fabric / Power BI Administrator** — both the gateway-clusters endpoint and the scanner require tenant-admin rights. (Running headless as a service principal / managed identity additionally needs the tenant setting *"Service principals can use Power BI APIs"* on, and the SP/MI assigned the **Fabric Administrator** role.)
3. Tenant setting **"Enhance admin APIs responses with detailed metadata"** should be **On** so `datasourceDetails` are returned.

## Temporal model
- **`gateways`** is an **SCD Type 2** dimension: `valid_from`, `valid_to` (null = current), `is_current`, and a `row_hash` change key. Each run closes out changed/removed gateways and inserts new versions via Delta `MERGE` — unchanged gateways don't churn.
- The bridge tables (`gateway_members`, `gateway_admins`, `gateway_datasources`, `gateway_semantic_model_map`) are **daily snapshots** partitioned by `snapshot_date`; re-running the same day replaces that day's partition.

```sql
-- gateway state as of any point in time
SELECT * FROM gateways
WHERE valid_from <= TIMESTAMP'2026-06-01' AND (valid_to IS NULL OR valid_to > TIMESTAMP'2026-06-01');
-- current only
SELECT * FROM gateways WHERE is_current = true;
-- a bridge table as of a date  (nearest snapshot on/before the target)
SELECT * FROM gateway_semantic_model_map
WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM gateway_semantic_model_map WHERE snapshot_date <= DATE'2026-06-01');
```

In [ ]:
# ── Config & auth ────────────────────────────────────────────────────────────
from datetime import datetime, timezone
import time, json
import requests
import notebookutils            # older runtimes: use `mssparkutils` instead

POWERBI_API = "https://api.powerbi.com/v1.0/myorg"
POWERBI_V2  = "https://api.powerbi.com/v2.0/myorg"   # admin gateway-clusters endpoint
FABRIC_API  = "https://api.fabric.microsoft.com/v1"

# Target lakehouse + output table names. Writing by OneLake path (below) means
# the lakehouse does NOT need to be attached as the notebook's default.
LAKEHOUSE_NAME     = "lh_fabric_management"
LAKEHOUSE_SCHEMA   = "fabricmanagement"   # writes to Tables/<schema>/<table>;
                                          # requires lh_fabric_management to be a
                                          # schema-enabled lakehouse. Set None for
                                          # a classic (non-schema) lakehouse.
TBL_GATEWAYS       = "gateways"
TBL_GW_MEMBERS     = "gateway_members"
TBL_GW_ADMINS      = "gateway_admins"
TBL_GW_DATASOURCES = "gateway_datasources"
TBL_GW_MODEL_MAP   = "gateway_semantic_model_map"

EXCLUDE_PERSONAL = True     # skip personal ("My workspace") workspaces
SCAN_BATCH       = 100      # max workspaces per getInfo call (API limit)
POLL_SECONDS     = 3        # scanStatus poll interval
POLL_MAX         = 200      # give up on a scan after POLL_MAX * POLL_SECONDS

# Naive-UTC snapshot stamp for every row written this run
SCAN_TS = datetime.now(timezone.utc).replace(tzinfo=None, microsecond=0)
print("Scan timestamp (UTC):", SCAN_TS.isoformat())

def _headers():
    # One Power BI token is accepted by both api.powerbi.com and api.fabric.microsoft.com.
    token = notebookutils.credentials.getToken("pbi")
    return {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

def _raise(resp):
    if resp.status_code in (401, 403):
        raise PermissionError(
            f"HTTP {resp.status_code} on {resp.url}\n"
            "  -> The identity needs Fabric-admin / Gateway.Read.All rights for this call.")
    resp.raise_for_status()

def _get(url, params=None):
    r = requests.get(url, headers=_headers(), params=params, timeout=120)
    _raise(r)
    return r.json()

def _post(url, params=None, body=None):
    r = requests.post(url, headers=_headers(), params=params, json=body, timeout=120)
    _raise(r)
    return r.json()

In [ ]:
# ── 1. Gateways (tenant-wide, admin gateway-clusters endpoint) ───────────────
# GET /v2.0/myorg/gatewayclusters (note: NO /me/) is the tenant-admin list that
# backs "Manage connections and gateways -> Tenant administration". It returns
# every cluster in the tenant when the caller is a Fabric/Power BI admin.
# (Undocumented but stable -- it is exactly what the portal itself calls. The
# documented /v1/gateways returns only clusters you are a *member* of, so a
# tenant admin who is not a gateway admin gets an empty list from it.)
def list_gateway_clusters():
    out, url = [], f"{POWERBI_V2}/gatewayclusters?$expand=memberGateways,permissions"
    while url:
        data = _get(url)
        out.extend(data.get("value", []))
        url = data.get("@odata.nextLink")
    return out

def _annotation(member):
    ann = member.get("annotation")
    if isinstance(ann, str):
        try:
            return json.loads(ann)
        except Exception:
            return {}
    return ann or {}

try:
    clusters = list_gateway_clusters()
except Exception as exc:
    print(f"WARNING: gateway-clusters call failed ({exc}). "
          "Continuing with gateway ids from the scan only.")
    clusters = []

gw_by_id = {c["id"]: c for c in clusters}          # cluster id -> cluster
member_to_cluster = {}                              # member id -> cluster id
for c in clusters:
    for m in (c.get("memberGateways") or []):
        member_to_cluster[m.get("id")] = c["id"]

def resolve_cluster(gateway_id):
    # Scanner reports the primary/cluster id; fall back through member ids.
    c = gw_by_id.get(gateway_id)
    if not c and gateway_id in member_to_cluster:
        c = gw_by_id.get(member_to_cluster[gateway_id])
    return c or {}

by_type = {}
for c in clusters:
    by_type[c.get("type")] = by_type.get(c.get("type"), 0) + 1
print(f"Gateway clusters in tenant: {len(clusters)}")
for t, n in sorted(by_type.items(), key=lambda x: str(x[0])):
    print(f"  {str(t):<12} {n}")

In [ ]:
# ── 2. Scan the tenant (getInfo -> poll -> scanResult), batched ──────────────
def modified_workspace_ids():
    params = {"excludePersonalWorkspaces": str(EXCLUDE_PERSONAL).lower()}
    data = _get(f"{POWERBI_API}/admin/workspaces/modified", params=params)
    items = data if isinstance(data, list) else data.get("value", [])
    return [w["id"] for w in items]

def scan_workspaces(ws_ids):
    # Returns (all workspace objects, {datasourceInstanceId -> instance}).
    workspaces, instances = [], {}
    for start in range(0, len(ws_ids), SCAN_BATCH):
        batch = ws_ids[start:start + SCAN_BATCH]
        scan = _post(
            f"{POWERBI_API}/admin/workspaces/getInfo",
            params={"lineage": "true", "datasourceDetails": "true",
                    "datasetSchema": "false", "datasetExpressions": "false",
                    "getArtifactUsers": "false"},
            body={"workspaces": batch},
        )
        scan_id = scan["id"]

        for _ in range(POLL_MAX):
            status = _get(f"{POWERBI_API}/admin/workspaces/scanStatus/{scan_id}").get("status")
            if status == "Succeeded":
                break
            if status in ("Failed", "Disabled"):
                raise RuntimeError(f"Scan {scan_id} ended with status={status}")
            time.sleep(POLL_SECONDS)
        else:
            raise TimeoutError(f"Scan {scan_id} did not finish after {POLL_MAX*POLL_SECONDS}s")

        result = _get(f"{POWERBI_API}/admin/workspaces/scanResult/{scan_id}")
        workspaces.extend(result.get("workspaces", []) or [])
        # Instances are deduped across batches; keyed by their datasourceId,
        # which is what a dataset's datasourceUsages reference.
        for inst in (result.get("datasourceInstances") or []):
            instances[inst.get("datasourceId")] = inst
        for inst in (result.get("misconfiguredDatasourceInstances") or []):
            instances.setdefault(inst.get("datasourceId"), inst)
        print(f"  batch {start//SCAN_BATCH + 1}: {len(batch):>3} workspaces, "
              f"{len(result.get('datasourceInstances') or []):>4} datasource instances")
    return workspaces, instances

ws_ids = modified_workspace_ids()
print(f"Workspaces to scan: {len(ws_ids)}")
scanned_workspaces, datasource_instances = scan_workspaces(ws_ids)
print(f"Datasource instances (deduped): {len(datasource_instances)}")

In [ ]:
# ── 3. Build the rows ────────────────────────────────────────────────────────
def _conn(inst):
    cd = inst.get("connectionDetails") or {}
    if isinstance(cd, str):
        try:
            cd = json.loads(cd)
        except Exception:
            cd = {}
    return cd

def _contact(ann):
    ci = ann.get("gatewayContactInformation")
    return ";".join(ci) if isinstance(ci, list) else ci

# 3a. gateways (per cluster) + members + admins --------------------------------
# source="api"  -> from the admin gateway-clusters endpoint
# source="scan" -> referenced by a model but absent from the clusters call (rare)
gateway_rows, member_rows, admin_rows = [], [], []
for c in clusters:
    members = c.get("memberGateways") or []
    primary = members[0] if members else {}
    pann = _annotation(primary)
    opts = c.get("options") or {}
    perms = c.get("permissions") or []
    gateway_rows.append({
        "gateway_id": c.get("id"),
        "gateway_name": c.get("name"),
        "gateway_type": c.get("type"),
        "member_count": len(members),
        "admin_count": sum(1 for p in perms if p.get("role") == "Admin"),
        "primary_status": primary.get("status"),
        "primary_version": primary.get("version"),
        "primary_machine": pann.get("gatewayMachine"),
        "contact_info": _contact(pann),
        "vnet_subnet_id": pann.get("gatewayVirtualNetworkSubnetId"),
        "cloud_datasource_refresh": opts.get("CloudDatasourceRefresh"),
        "custom_connectors": opts.get("CustomConnectors"),
        "source": "api",
        "scan_timestamp": SCAN_TS,
    })
    for m in members:
        mann = _annotation(m)
        member_rows.append({
            "gateway_id": c.get("id"),
            "gateway_name": c.get("name"),
            "member_id": m.get("id"),
            "member_name": m.get("name"),
            "status": m.get("status"),
            "state": m.get("state"),
            "version": m.get("version"),
            "version_status": m.get("versionStatus"),
            "update_status": m.get("onPremGatewayUpdateStatus"),
            "machine": mann.get("gatewayMachine"),
            "department": mann.get("gatewayDepartment"),
            "contact_info": _contact(mann),
            "vnet_subnet_id": mann.get("gatewayVirtualNetworkSubnetId"),
            "expiry_date": str(m.get("expiryDate")) if m.get("expiryDate") else None,
            "scan_timestamp": SCAN_TS,
        })
    for p in perms:
        admin_rows.append({
            "gateway_id": c.get("id"),
            "gateway_name": c.get("name"),
            "principal_id": p.get("id"),
            "principal_type": p.get("principalType"),
            "role": p.get("role"),
            "tenant_id": p.get("tenantId"),
            "scan_timestamp": SCAN_TS,
        })

# Any gateway id a model references but the clusters call didn't return.
api_ids = set(gw_by_id) | set(member_to_cluster)
scan_gw_ids = {i.get("gatewayId") for i in datasource_instances.values() if i.get("gatewayId")}
for gid in sorted(scan_gw_ids - api_ids):
    gateway_rows.append({
        "gateway_id": gid, "gateway_name": None, "gateway_type": None,
        "member_count": None, "admin_count": None, "primary_status": None,
        "primary_version": None, "primary_machine": None, "contact_info": None,
        "vnet_subnet_id": None, "cloud_datasource_refresh": None,
        "custom_connectors": None, "source": "scan", "scan_timestamp": SCAN_TS,
    })

print(f"gateway clusters ....... {len(clusters)}")
print(f"gateway members ........ {len(member_rows)}")
print(f"gateway admins ......... {len(admin_rows)}")
print(f"scanner-only gateways .. {len(scan_gw_ids - api_ids)}")

# 3b. distinct gateway-bound data sources (from the scanner) --------------------
gw_datasource_rows = []
for inst_id, inst in datasource_instances.items():
    gw_id = inst.get("gatewayId")
    if not gw_id:                       # cloud source, not fronted by a gateway
        continue
    c = resolve_cluster(gw_id)
    cd = _conn(inst)
    gw_datasource_rows.append({
        "gateway_id": c.get("id") or gw_id,
        "gateway_name": c.get("name"),
        "gateway_type": c.get("type"),
        "datasource_id": inst_id,
        "datasource_type": inst.get("datasourceType"),
        "datasource_server": cd.get("server") or cd.get("url") or cd.get("path"),
        "datasource_database": cd.get("database"),
        "connection_details": json.dumps(cd) if cd else None,
        "scan_timestamp": SCAN_TS,
    })

# 3c. gateway <-> semantic model map -------------------------------------------
map_rows = []
for ws in scanned_workspaces:
    ws_id, ws_name = ws.get("id"), ws.get("name")
    for ds in (ws.get("datasets") or []):            # datasets == semantic models
        good = ds.get("datasourceUsages") or []
        bad  = ds.get("misconfiguredDatasourceUsages") or []
        mis_ids = {u.get("datasourceInstanceId") for u in bad}
        seen = set()
        for u in good + bad:
            inst_id = u.get("datasourceInstanceId")
            if inst_id in seen:
                continue
            seen.add(inst_id)
            inst = datasource_instances.get(inst_id)
            if not inst or not inst.get("gatewayId"):   # skip non-gateway sources
                continue
            c = resolve_cluster(inst["gatewayId"])
            cd = _conn(inst)
            map_rows.append({
                "gateway_id": c.get("id") or inst["gatewayId"],
                "gateway_name": c.get("name"),
                "gateway_type": c.get("type"),
                "datasource_id": inst_id,
                "datasource_type": inst.get("datasourceType"),
                "datasource_server": cd.get("server") or cd.get("url") or cd.get("path"),
                "datasource_database": cd.get("database"),
                "connection_details": json.dumps(cd) if cd else None,
                "workspace_id": ws_id,
                "workspace_name": ws_name,
                "semantic_model_id": ds.get("id"),
                "semantic_model_name": ds.get("name"),
                "is_misconfigured": inst_id in mis_ids,
                "scan_timestamp": SCAN_TS,
            })

print(f"gateway-bound datasources .......... {len(gw_datasource_rows)}")
print(f"gateway <-> semantic-model links ... {len(map_rows)}")

In [ ]:
# ── 4. Write to lh_fabric_management (SCD2 dim + daily-snapshot bridges) ──────
# gateways      -> SCD Type 2 (valid_from / valid_to / is_current + row_hash)
# gateway_*     -> daily snapshot (partitioned by snapshot_date)
from delta.tables import DeltaTable
from pyspark.sql import functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, BooleanType, TimestampType)

SNAP_DATE = SCAN_TS.date()

def _rows(dicts, schema):
    # dict -> tuple in schema order; also yields a valid empty DataFrame.
    names = [f.name for f in schema.fields]
    return [tuple(d.get(n) for n in names) for d in dicts]

def _tables_path(lakehouse_name):
    # OneLake Tables/ path for the target lakehouse. Writing to an absolute path
    # avoids "No default context found" when no default lakehouse is attached.
    lh = notebookutils.lakehouse.get(lakehouse_name)
    props = (lh.get("properties") or {}) if isinstance(lh, dict) else {}
    return (props.get("oneLakeTablesPath")
            or f'{props.get("abfsPath", "").rstrip("/")}/Tables')

TABLES_PATH = _tables_path(LAKEHOUSE_NAME)

def _table_uri(name):
    sub = name if not LAKEHOUSE_SCHEMA else f"{LAKEHOUSE_SCHEMA}/{name}"
    return f"{TABLES_PATH}/{sub}"

def _df(dicts, schema):
    return spark.createDataFrame(_rows(dicts, schema), schema=schema)

def _with_hash(df, cols):
    parts = [F.coalesce(F.col(c).cast("string"), F.lit("<null>")) for c in cols]
    return df.withColumn("row_hash", F.sha2(F.concat_ws("||", *parts), 256))

def write_scd2(dicts, schema, name, keys):
    # SCD Type 2 by natural key. row_hash is computed over the business columns
    # only (schema has no scan_timestamp), so unchanged rows don't churn.
    path = _table_uri(name)
    biz = [f.name for f in schema.fields]
    src = _with_hash(_df(dicts, schema), biz)
    if not DeltaTable.isDeltaTable(spark, path):
        (src.withColumn("valid_from", F.lit(SCAN_TS).cast("timestamp"))
            .withColumn("valid_to",  F.lit(None).cast("timestamp"))
            .withColumn("is_current", F.lit(True))
            .write.format("delta").save(path))
        print(f"  {src.count():>5} rows -> {name} (initial SCD2 load)")
        return
    dt = DeltaTable.forPath(spark, path)
    cond = " AND ".join(f"t.{k} = s.{k}" for k in keys)
    # 1) close rows that changed (hash differs) or disappeared (not in source)
    (dt.alias("t").merge(src.alias("s"), f"({cond}) AND t.is_current = true")
       .whenMatchedUpdate(condition="t.row_hash <> s.row_hash",
                          set={"is_current": F.lit(False), "valid_to": F.lit(SCAN_TS)})
       .whenNotMatchedBySourceUpdate(condition="t.is_current = true",
                          set={"is_current": F.lit(False), "valid_to": F.lit(SCAN_TS)})
       .execute())
    # 2) insert a fresh current version for new + changed keys
    cur = dt.toDF().where("is_current = true").select(*keys)
    ins = src.join(cur, keys, "left_anti")
    (ins.withColumn("valid_from", F.lit(SCAN_TS).cast("timestamp"))
        .withColumn("valid_to",  F.lit(None).cast("timestamp"))
        .withColumn("is_current", F.lit(True))
        .write.format("delta").mode("append").save(path))
    print(f"  +{ins.count()} new/changed versions -> {name} (SCD2 merge)")

def write_snapshot(dicts, schema, name):
    # Full daily snapshot; re-running the same day replaces that partition.
    path = _table_uri(name)
    df = _df(dicts, schema).withColumn("snapshot_date", F.lit(SNAP_DATE))
    w = (df.write.format("delta").partitionBy("snapshot_date")
           .option("overwriteSchema", "true").mode("overwrite"))
    if DeltaTable.isDeltaTable(spark, path):
        w = w.option("replaceWhere", f"snapshot_date = '{SNAP_DATE}'")
    w.save(path)
    print(f"  {df.count():>5} rows -> {name} (snapshot {SNAP_DATE})")

gateways_schema = StructType([
    StructField("gateway_id", StringType()),
    StructField("gateway_name", StringType()),
    StructField("gateway_type", StringType()),
    StructField("member_count", IntegerType()),
    StructField("admin_count", IntegerType()),
    StructField("primary_status", StringType()),
    StructField("primary_version", StringType()),
    StructField("primary_machine", StringType()),
    StructField("contact_info", StringType()),
    StructField("vnet_subnet_id", StringType()),
    StructField("cloud_datasource_refresh", BooleanType()),
    StructField("custom_connectors", BooleanType()),
    StructField("source", StringType()),
])

members_schema = StructType([
    StructField("gateway_id", StringType()),
    StructField("gateway_name", StringType()),
    StructField("member_id", StringType()),
    StructField("member_name", StringType()),
    StructField("status", StringType()),
    StructField("state", StringType()),
    StructField("version", StringType()),
    StructField("version_status", StringType()),
    StructField("update_status", StringType()),
    StructField("machine", StringType()),
    StructField("department", StringType()),
    StructField("contact_info", StringType()),
    StructField("vnet_subnet_id", StringType()),
    StructField("expiry_date", StringType()),
    StructField("scan_timestamp", TimestampType()),
])

admins_schema = StructType([
    StructField("gateway_id", StringType()),
    StructField("gateway_name", StringType()),
    StructField("principal_id", StringType()),
    StructField("principal_type", StringType()),
    StructField("role", StringType()),
    StructField("tenant_id", StringType()),
    StructField("scan_timestamp", TimestampType()),
])

datasources_schema = StructType([
    StructField("gateway_id", StringType()),
    StructField("gateway_name", StringType()),
    StructField("gateway_type", StringType()),
    StructField("datasource_id", StringType()),
    StructField("datasource_type", StringType()),
    StructField("datasource_server", StringType()),
    StructField("datasource_database", StringType()),
    StructField("connection_details", StringType()),
    StructField("scan_timestamp", TimestampType()),
])

map_schema = StructType([
    StructField("gateway_id", StringType()),
    StructField("gateway_name", StringType()),
    StructField("gateway_type", StringType()),
    StructField("datasource_id", StringType()),
    StructField("datasource_type", StringType()),
    StructField("datasource_server", StringType()),
    StructField("datasource_database", StringType()),
    StructField("connection_details", StringType()),
    StructField("workspace_id", StringType()),
    StructField("workspace_name", StringType()),
    StructField("semantic_model_id", StringType()),
    StructField("semantic_model_name", StringType()),
    StructField("is_misconfigured", BooleanType()),
    StructField("scan_timestamp", TimestampType()),
])

print(f"Writing to {LAKEHOUSE_NAME} (schema={LAKEHOUSE_SCHEMA}): {TABLES_PATH}")
write_scd2(gateway_rows, gateways_schema, TBL_GATEWAYS, ["gateway_id"])   # SCD2 dim
write_snapshot(member_rows,        members_schema,     TBL_GW_MEMBERS)     # daily snapshot
write_snapshot(admin_rows,         admins_schema,      TBL_GW_ADMINS)
write_snapshot(gw_datasource_rows, datasources_schema, TBL_GW_DATASOURCES)
write_snapshot(map_rows,           map_schema,         TBL_GW_MODEL_MAP)

## Verify

A quick roll-up: semantic models and workspaces per gateway.

In [ ]:
# Read back by path (no default lakehouse required) and roll up the latest snapshot.
spark.read.format("delta").load(_table_uri(TBL_GW_MODEL_MAP)).createOrReplaceTempView("gw_model_map")
q = (
    "SELECT gateway_type, "
    "       COALESCE(gateway_name, gateway_id) AS gateway, "
    "       COUNT(DISTINCT semantic_model_id) AS semantic_models, "
    "       COUNT(DISTINCT workspace_id)      AS workspaces, "
    "       COUNT(DISTINCT datasource_id)     AS datasources "
    "  FROM gw_model_map "
    " WHERE snapshot_date = (SELECT MAX(snapshot_date) FROM gw_model_map) "
    " GROUP BY gateway_type, COALESCE(gateway_name, gateway_id) "
    " ORDER BY semantic_models DESC"
)
display(spark.sql(q))
print("\\nCurrent gateways (SCD2 is_current):")
spark.read.format("delta").load(_table_uri(TBL_GATEWAYS)).createOrReplaceTempView("gateways_scd2")
display(spark.sql(
    "SELECT gateway_type, COUNT(*) AS current_gateways "
    "FROM gateways_scd2 WHERE is_current = true GROUP BY gateway_type ORDER BY 2 DESC"))